# Job Listing Prioritization for Career Changers: EDA/SDA 3 (Text features)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

**What do the listing descriptions say?** This notebook turns the description text of EDA_SDA_2's scope listings into **features**, and reports the text rules as **keyword rules with their firing rates**, so `EDA_SDA_4` can test every feature (text included) and the app can label what a listing asks and offer opt-in filters. **Decided 2026-10-13: the validation samples (section 5) will not be hand-checked,** so the text blockers (years asked, degree required, clearance, no sponsorship and the other career-change flags) are **unvalidated**. They are labels on each listing and filters a career changer turns on (for example "I have no degree"), never hidden by default. The same decision covers EDA_SDA_2's two title rules (staffing agency from recruiter titles; the 13-1161 split into analysts and marketing specialists): they are not hand-checked here either, and are labels and opt-in filters. Only reliable rules hide by default (seniority, likely ghosts, occupations that aren't a tech path); every rule reports how many listings it would hide. **It builds no matching models:** résumé-to-listing fit (keyword overlap, TF-IDF, BGE embeddings, RRF) is in `ML_Fit`, which uses this notebook's cleaned text and skills. A text feature is kept only if it adds information beyond the non-text features.

**For career changers,** the most important text features are what a listing asks of a newcomer (experience, education, licenses, training) and which of its skills carry over from other fields.

**Inputs:** EDA_SDA_2's scope table, plus the description text for those listings, loaded from the saved flattened parts (EDA_SDA_1's table keeps only description features, to fit in memory).

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, unvalidated text rules as labels and opt-in filters, skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

**View:** listings with a description (82% of the scope; EDA_SDA_2 3.9). Text features are **unknown** for the rest, never "no", and "has a description" stays a standard confounder: listings without one close in about 3 days.

**Data rules:** aggregates only; never print descriptions, titles or sample rows, except in cells tagged `data-rows` (review cells for a teammate to check locally; cleared before every commit). Text stays local: no listing text goes to outside AI services. Clear outputs before committing.

### Clarifying questions
- What does description length actually capture: employer intent, or boilerplate?
- Does urgency wording, read in context rather than by keyword, relate to how long listings stay open? (EDA_SDA_1 left it here: its keyword flag misfired and showed no gap.)
- Can evergreen wording ("always accepting applications") identify ghost listings?
- Are EDA_SDA_1's career-change flags (years required, degree required, training and career-changer wording) accurate, and what do they miss? Two are urgent: EDA_SDA_1's "asks 5+ years" rule would hide 35% of open listings if a user turned it on, and its degree flag fires on only 3.7% (both measured in 3a and 3b below; EDA_SDA_2 3.11, on its own screened view, quotes 32% and 2.6%).
- Which requirements are labels and opt-in filters (years asked, degree, clearance, sponsorship, certification), and how many listings would each hide if a user turned it on? (Not validated: see the 2026-10-13 decision above.)
- Which skills does each listing ask for, mapped to O*NET skills, so `ML_Fit` can match on skills rather than words?

### Roadmap

1. **Setup:** the scope table and its description text
2. **Description cleanup:** HTML to structured text, boilerplate removal, normalization and lemmas
3. **The text rules:** years of experience (required vs. preferred) and degree requirements, kept as labels and opt-in filters
4. **Other text features:** evergreen wording, hiring-difficulty phrases, licenses and certifications, training and career-changer wording, urgency, remote, length
5. **Validation samples:** written, but not being marked (decided 2026-10-13); rules are reported as keyword rules with their firing rates
6. **Skills mapped to O*NET** (for `ML_Fit`)
7. **Final table:** text rules saved as labels with the count each would hide (nothing hidden by default, nothing removed), for EDA_SDA_4 and `ML_Fit`
8. **Conclusion**


## Plan: text features to build

| Feature | Question | Notes |
|:--|:--|:--|
| **Description cleanup** (moved here from EDA_SDA_2 2b) | Strip HTML and boilerplate (equal-opportunity statements, benefits, "about us") | `desc_clean` and `desc_words_clean`; the base for every feature below and for `ML_Fit` |
| Description length | Is it employer intent, or just boilerplate? | Measured on `desc_clean` |
| Urgency wording | Does real urgency (in context, not "urgent care") go with shorter or longer windows? | EDA_SDA_1's keyword flag showed no gap; may mark hard-to-fill roles |
| Remote mentions | Does remote add beyond `contract_types`? | |
| **Evergreen wording** | Listings that collect applications instead of filling one job: a ghost-listing signal | Six phrase groups, each scored and checked on its own: **talent pool** ("join our talent community", "future opportunities"), **no specific opening** ("general application", "not for a specific position"), **always hiring** ("always accepting applications", "rolling basis"), **many openings in one ad** ("multiple openings", "various locations"), **pipeline labels** ("pipeline requisition", "evergreen requisition"), **staffing-style** ("on behalf of our client"); plus a generic-title check ("Software Engineer – All Levels"). Compared with EDA_SDA_2's likely-ghost flag (3.1): kept only if it adds to "old and from a quiet company" |
| Hiring-difficulty phrases | "Sign-on bonus", "hiring immediately", "relocation assistance", "will train" | Pairs with reposts and listing age |
| Years of experience required | "3+ years": is EDA_SDA_1's rule accurate? Required or preferred? In the field, or any experience? | The most common blocker for career changers |
| Education required | "Bachelor's required" vs. "or equivalent experience"; field-specific degrees | EDA_SDA_1's pattern only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matches: it flags only 3.7% of listings. Rebuilt in section 3, with requirements and preferred sections told apart |
| **Clearance, sponsorship, travel, shifts** (added 2026-10-10) | Security clearance required; "no visa sponsorship"; travel percentage; shift or on-call work; contract-to-hire | Clearance and sponsorship are blockers for some career changers: a label on the listing and an opt-in filter in the app, like a missing degree (never hidden by default) |
| Licenses and certifications required | "Security+", "CCNA", "CISSP" and the like | A required credential the career changer lacks: a label and an opt-in filter |
| Training and career-changer wording | "Will train", "paid training", "career changers welcome", "transferable skills" | Reports EDA_SDA_1's flags as keyword rules with their firing rates (the hand-check sample is not being marked, decided 2026-10-13) |
| **Text-rule filters** | Which years-asked threshold and which required credentials would a user filter on, per role bucket? | Not validated (decided 2026-10-13): saved in this notebook's final table as labels, with a count of the listings each would hide if turned on (opt-in in the app, never hidden by default, never removed) |
| Required vs. preferred qualifications | Split on headings like "Requirements" and "Nice to have" | Part of fit and gap analysis in `ML_Fit` |
| **Skills, mapped to O*NET** | Extracted with spaCy (PredictLeads `tags` are sparse) and mapped to O*NET skills | `ML_Fit` matches résumés to listings on these skills, so "transferable" doesn't depend on the old field's vocabulary |

**Moved to `ML_Fit`:** role similarity, TF-IDF and embeddings for matching. **Moved to `ML_Rank`:** TF-IDF terms that predict how fast a listing closes (a text input to the timing model). Topic modeling stays a stretch goal.

## Plan: preprocessing

Course standard (MLforTexts): normalize text (lowercase, regex to keep letters), remove NLTK stopwords, lemmatize with spaCy. Advanced: remove boilerplate where headings make it detectable. Any vocabulary learned from the text is fit on the training months only.

**Evaluation (course standard):** the plan was precision and recall on a hand-checked random sample against EDA_SDA_1's keyword rule. **Decided 2026-10-13: the samples will not be hand-checked,** so each rule is reported as a keyword rule with its firing rate, against EDA_SDA_1's rule where one exists, and not as a validated blocker.


## 1. Setup

In [ ]:
# 1a Libraries, paths and the shared code: src/text_features.py does the cleanup and the text rules batch by batch,
# so the full scope's descriptions never sit in memory at once.
import gc
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, INK, MUTED, CLEAN_DIR, PART2_TABLE, PART3_TABLE,
    bar_h, finish, show, show_shaded, use_chart_style,
)
from src import text_features as tf
use_chart_style()

TRAIN_END = "2026-07"  # anything learned from the text (template lines) is learned on April–July only, ML_Rank's training months
DESC_PATH = CLEAN_DIR / "scope_descriptions.parquet"  # EDA_SDA_1 8d: id, description (HTML)
FEATURES_PATH = CLEAN_DIR / "eda3_text_features.parquet"  # every listing's text features (cache; delete to rebuild)
CLEAN_PATH = CLEAN_DIR / "eda3_desc_clean.parquet"  # every listing's cleaned text, for ML_Fit
LEMMA_PATH = CLEAN_DIR / "eda3_lemmas_sample.parquet"  # lemmas for a stratified sample (2c)
REMOVED_PATH = CLEAN_DIR / "eda3_removed_words.json"  # words removed per boilerplate reason (2b)
VALIDATION_DIR = CLEAN_DIR / "eda3_validation"  # hand-check samples (section 5); local only, like all of data/
TEMPLATE_SAMPLE = 120_000  # listings used to learn shared template lines (training months, with a description)
LEMMA_SAMPLE = 150_000  # listings lemmatized with spaCy (stratified by role bucket)
_t0 = time.time()

In [ ]:
# 1b Load only the columns this notebook needs from EDA_SDA_2's scope table (the text itself is read batch by batch
# in section 2). Only counts are shown.
COLS = ["id", "company_id", "first_seen", "month", "role_bucket", "tripleten_program", "has_desc", "live", "days_open",
        "volume_band", "source_type", "likely_ghost", "stale", "cc_visible", "level", "is_repost", "hidden_occupation", "career_open",
        "years_required", "degree_required", "equivalent_experience", "entry_wording", "changer_wording", "urgency", "mentions_remote"]
jobs = pd.read_parquet(PART2_TABLE, columns=COLS)
jobs["month"] = jobs.month.astype(str)
meta = jobs.set_index("id")[["company_id", "month"]]
show("1b The scope table and its text", pd.DataFrame({"value": {
    "listings (EDA_SDA_2 scope table)": f"{len(jobs):,}",
    "with a description (has_desc)": f"{jobs.has_desc.mean():.1%}",
    "descriptions file": f"{DESC_PATH.name}: {pq.ParquetFile(DESC_PATH).metadata.num_rows:,} rows, {DESC_PATH.stat().st_size / 1e9:.2f} GB",
    "first seen in the training months (April–July)": f"{(jobs.month <= TRAIN_END).sum():,}",
}}))

## 2. Description cleanup

**Why:** every text feature, and `ML_Fit`'s matching, should read what the listing asks for, not the legal and benefits text most listings share. At full scale the descriptions are several GB of text, so **the cleanup runs batch by batch** (`src/text_features.py`): each batch is cleaned, its features computed and both written to disk, then dropped from memory.

| Step | What | Output |
|:--|:--|:--|
| 2a | HTML to text, one line per paragraph, list item or heading (long unbroken text split into sentences) | Lines per listing |
| 2b | Remove boilerplate line by line: known patterns (equal opportunity and legal, benefits, application logistics), "About the company" sections, and text shared word for word across 5+ employers (learned on a sample of April–July listings; lines with a requirement cue are never cut) | `desc_clean` (`eda3_desc_clean.parquet`), `words_clean` |
| 2c | Course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas, on a stratified sample | `desc_lemmas` (for word counts and TF-IDF later) |

The rules in sections 3–4 run on the cleaned text, which keeps case, numbers and punctuation ("3+ years", "B.S.", "Security+"), and tell **requirement lines from preferred ones** using the headings kept in 2a.

In [ ]:
# 2a–2b Learn shared template lines on a sample of training-month listings, then clean every description and compute
# its text features, batch by batch on several processes (cached: delete eda3_text_features.parquet to rebuild).
have = jobs[jobs.has_desc & (jobs.month <= TRAIN_END)]
sample_ids = set(have.sample(min(TEMPLATE_SAMPLE, len(have)), random_state=42).id)
if tf.cache_valid(FEATURES_PATH, DESC_PATH, REMOVED_PATH, CLEAN_PATH):  # an unreadable file (a killed run) means rebuild
    removed = json.loads(REMOVED_PATH.read_text())
    print(f"loaded the text features from {FEATURES_PATH.name} (delete it to rebuild 2a–2b)")
else:
    templates = tf.learn_templates(DESC_PATH, sample_ids, meta.company_id)  # streams batches on several processes; keeps only line keys
    print(f"shared template lines learned on {len(sample_ids):,} April–July listings: {len(templates):,}")
    start = time.time()
    removed = tf.build(DESC_PATH, meta, templates, FEATURES_PATH, CLEAN_PATH,
                       progress=lambda n, k: print(f"  batch {n + 1}: {k:,} listings, {time.time() - start:.0f}s", end="\r"))
    REMOVED_PATH.write_text(json.dumps(removed))
    print(f"\ncleaned and featurized every description in {(time.time() - start) / 60:.1f} min")
text = pd.read_parquet(FEATURES_PATH, dtype_backend="numpy_nullable")  # blanks stay unknown (NA), not None in object columns
jobs = jobs.merge(text, on="id", how="left", suffixes=("_eda1", ""))
has_text = jobs.words_raw.fillna(0) > 0
by_reason = pd.Series(removed, dtype=float).sort_values(ascending=False)
show("2b Words removed as boilerplate (listings with a description)", pd.DataFrame({
    "words removed": by_reason,
    "share of all description words": by_reason / jobs.words_raw.sum(),
}).round(3))
src = jobs[has_text].groupby("source_type")
show("2b Share of each description kept after cleanup, by source (median per listing)", pd.DataFrame({
    "median words before": src.words_raw.median(), "median words after": src.words_clean.median(),
    "median share kept": src.apply(lambda g: (g.words_clean / g.words_raw).median(), include_groups=False),
    "cleaned to nothing": src.apply(lambda g: g.words_clean.eq(0).mean(), include_groups=False),
    "listings": src.size(),
}).round(3))

**What the data shows**
- **The cleanup ran on all 908,291 scope listings, 82.4% of which have a description,** in 15.6 minutes (batch by batch). The shared template lines were learned on 120,000 April–July listings and found 6,556.
- **Boilerplate removed, by kind (words, share of all description words):** equal opportunity and legal 36.1M (6.6%), about the company 24.1M (4.4%), benefits 23.9M (4.4%), shared template text 20.4M (3.7%), application logistics 2.5M (0.40%).
- **Listings keep most of their text:** the median listing keeps 85–86% of its words in every source type (company site 716 to 559 words, hiring platform 784 to 617, job board 620 to 496), and none is cleaned to nothing.

In [ ]:
# 2b review: a few random listings after cleanup, for a teammate to check locally that 2b keeps the requirements.
# Shows real listing text. Tagged "data-rows": clear this output before committing.
clean_sample = pq.ParquetFile(CLEAN_PATH).read_row_group(0).to_pandas().sample(3, random_state=7)
for t in clean_sample.desc_clean:
    print("=" * 100, "\n", t[:2500])

In [ ]:
# 2c Normalize and lemmatize (course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas) on a stratified
# sample, since spaCy on every listing would take hours. desc_lemmas feeds word counts and TF-IDF in ML_Fit and ML_Rank.
import nltk
import spacy
try:
    STOP = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)  # one-time download of NLTK's word list (no project data leaves the laptop)
    STOP = set(nltk.corpus.stopwords.words("english"))

if LEMMA_PATH.exists():
    lem = pd.read_parquet(LEMMA_PATH)
    print(f"loaded {len(lem):,} lemmatized listings from {LEMMA_PATH.name}")
else:
    pool = jobs[has_text]
    lem_ids = set(pool.groupby("role_bucket").sample(frac=min(LEMMA_SAMPLE / len(pool), 1), random_state=42).id)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # the tagger and lemmatizer are all that's needed
    parts = []
    for batch in pq.ParquetFile(CLEAN_PATH).iter_batches(batch_size=20_000):
        d = batch.to_pandas()
        d = d[d.id.isin(lem_ids)]
        lemmas = [" ".join(t.lemma_.lower() for t in doc if t.is_alpha and t.lemma_.lower() not in STOP)
                  for doc in nlp.pipe(d.desc_clean.str.replace("\n", ". ").str[:20000], batch_size=64)]
        parts.append(pd.DataFrame({"id": d.id.to_numpy(), "desc_lemmas": lemmas}))
    lem = pd.concat(parts, ignore_index=True)
    lem.to_parquet(LEMMA_PATH, index=False)
    print(f"saved {len(lem):,} lemmatized listings to {LEMMA_PATH.name}")
words_lem = lem.desc_lemmas.str.split().str.len()
show("2c Words per description at each step (listings with a description)", pd.DataFrame({
    "after HTML removal": jobs.words_raw[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "after boilerplate removal": jobs.words_clean[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "lemmas, stopwords removed (sample)": words_lem.describe(percentiles=[0.1, 0.5, 0.9]),
}).drop(index="count").round(0))

In [ ]:
# 2 Chart: words per description at each cleanup step (left), and what boilerplate removal takes out (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
bins = np.logspace(0, np.log10(max(jobs.words_raw.max(), 10)), 40)
for vals, label, color in [(jobs.words_raw[has_text], "after HTML removal", MUTED), (jobs.words_clean[has_text], "after boilerplate removal", BLUE),
                           (words_lem, "lemmas, stopwords removed (sample)", ORANGE)]:
    ax1.hist(vals.clip(lower=1), bins=bins, histtype="step", lw=1.8, color=color, label=label, density=True)
ax1.set(xscale="log", title="Words per description at each step", xlabel="words (log scale)", ylabel="share of listings (density)")
ax1.legend(fontsize=8, frameon=False)
bar_h(ax2, 100 * by_reason / jobs.words_raw.sum(), "Share of all description words removed", "% of words", "{:.1f}%")
finish()

**What the data shows**
- **The chart:** the word counts are two humps on the log scale, a small group of very short descriptions (under about 60 words) and the main body at about 300-1,200 words; boilerplate removal shifts the main body left without changing its shape, and the lemma step moves it to about 250-500 words, leaving a spike of near-empty sampled listings at 1 word. On the right, equal opportunity and legal text is the largest block removed (6.6% of all words), then about the company and benefits (4.4% each), shared template text (3.7%) and application logistics (0.4%).
- **Words per description fall at each step:** a median of 735 after HTML removal, 573 after boilerplate removal and 394 as lemmas with stopwords removed (the lemma step is on a 150,001-listing sample). Means are 732, 590 and 406.
- **The longest descriptions are long before and after:** the 90th percentile is 1,149 words after HTML removal and 957 after boilerplate removal, and the maximum (7,827 words) is the same at both steps.

## 3. The blocker rules: years of experience and degree

EDA_SDA_1's rules read every line the same way: "3+ years required" and "5 years preferred" both count, and its degree rule only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matched (3.7% of listings). Here both are rebuilt on the cleaned text, with **requirement lines told apart from preferred ones** (from the headings kept in 2a, and lines that say "preferred" or "a plus"):

- **Years:** `years_asked` (any line, like EDA_SDA_1), `years_required` (requirement lines only), `years_preferred`. Also reads "minimum of two years" and number words.
- **Degree:** `degree_required` (a degree named on a requirement line), `degree_preferred`, `degree_equivalent` ("or equivalent experience", "in lieu of a degree"), `degree_level` (the highest level required).

The question for the app: **how many listings would each rule hide if a user turned it on** (too many years required; a degree they don't have). These are unvalidated keyword rules (decided 2026-10-13), so they label listings and drive opt-in filters; nothing hides by default.

In [ ]:
# 3a Years of experience: the new rules vs. EDA_SDA_1's, and how many listings each would hide at 5+ years
def share(mask):
    """Share of rows where mask is true; a blank (no years stated) counts as false, so every share is over all rows."""
    return mask.fillna(False).astype(bool).mean()

d = jobs[has_text]
y_old, y_new = d.years_required_eda1.astype(float), d.years_asked.astype(float)
agree = pd.Series({
    "both say no years": share(y_old.isna() & y_new.isna()),
    "same number": share(y_old == y_new),
    "new rule finds more years (or finds years where EDA_SDA_1 found none)": share(y_new > y_old.fillna(0)),
    "new rule finds fewer (or none where EDA_SDA_1 found some)": share(y_new.fillna(0) < y_old),
}, name="share of listings with a description")
show("3a EDA_SDA_1's years rule vs. the new one (any line)", agree.to_frame())
open_now = d.live
y_req, y_pref = d.years_required.astype(float), d.years_preferred.astype(float)
rules = {
    "EDA_SDA_1: 5+ years anywhere": (y_old >= 5),
    "new: 5+ years anywhere": (y_new >= 5),
    "new: 5+ years on a requirement line": (y_req >= 5),
    "new: 5+ years only on preferred lines": (y_pref >= 5) & ~(y_req >= 5),
}
show("3a The 5+ years rule: share of listings with a description, and of open ones (what it would hide)", pd.DataFrame({
    "all listings": {k: share(v) for k, v in rules.items()},
    "open on September 30": {k: share(v[open_now]) for k, v in rules.items()},
}).round(3))
by_bucket = d.assign(y_new=y_new, y_req=y_req, y_old=y_old).groupby("role_bucket").agg(
    states_years=("y_new", lambda s: s.notna().mean()),
    median_required=("y_req", "median"),
    required_5plus=("y_req", lambda s: (s >= 5).mean()),
    required_2_or_fewer=("y_req", lambda s: (s <= 2).mean()),
    eda1_5plus=("y_old", lambda s: (s >= 5).mean()),
    listings=("id", "size"),
).sort_values("required_5plus")
show("3a Years required by role bucket (listings with a description)", by_bucket.round(3))

In [ ]:
# 3b Degree: the new rule vs. EDA_SDA_1's, with "or equivalent experience" and the level asked
d = jobs[has_text]
show("3b Degree rules: share of listings with a description", pd.Series({
    "EDA_SDA_1: degree required": d.degree_required_eda1.mean(),
    "new: a degree mentioned anywhere": d.degree_mentioned.mean(),
    "new: degree on a requirement line": d.degree_required.mean(),
    "new: degree only on preferred lines": (d.degree_preferred & ~d.degree_required).mean(),
    "new: or equivalent experience accepted": d.degree_equivalent.mean(),
    "new: degree required, no equivalent (the per-applicant blocker)": (d.degree_required & ~d.degree_equivalent).mean(),
}, name="share").round(3).to_frame())
show("3b Highest degree level on a requirement line", d.degree_level.value_counts(normalize=True, dropna=False).rename("share").round(3).to_frame())
show("3b Degree required (no equivalent) by role bucket", d.groupby("role_bucket").apply(
    lambda g: pd.Series({"degree required": g.degree_required.mean(), "no equivalent": (g.degree_required & ~g.degree_equivalent).mean(),
                         "bachelor's or higher": g.degree_level.isin(["bachelor", "master", "phd"]).mean()}),
    include_groups=False).sort_values("no equivalent").round(3))

In [ ]:
# 3 Chart: share asking 5+ years on a requirement line vs. EDA_SDA_1's rule, by role bucket
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(by_bucket))
ax.barh(y - 0.2, 100 * by_bucket.eda1_5plus, height=0.4, color=MUTED, label="EDA_SDA_1: 5+ years anywhere")
ax.barh(y + 0.2, 100 * by_bucket.required_5plus, height=0.4, color=BLUE, label="new: 5+ years on a requirement line")
ax.set(yticks=y, yticklabels=by_bucket.index, xlabel="% of listings with a description", title="Who asks 5+ years: old rule vs. requirement lines only")
ax.legend(fontsize=8, frameon=False)
finish()

**What the data shows**
- **The chart:** the requirement-line rule finds 5+ years in more listings than EDA_SDA_1's rule in every bucket but business and data analysis (about 24% vs. 26%), most in security (about 39% to 43%), networks and systems, other computer roles and quality assurance; the order barely changes, from IT project management (about 49%) at the top to IT support (about 14%) at the bottom.
- **EDA_SDA_1's years rule was too narrow:** it finds 5+ years in 33.7% of listings with a description, the rebuilt rule in 39.1% (any line). The new rule finds more years in 12% of listings and fewer in 1.2%; both say no years in 34%.
- **Required vs. preferred matters a little for years:** 5+ years on a requirement line is 35.6% of listings (36% of those open on September 30), and 3.4% ask it only on preferred lines. A filter on 5+ years would hide about a third of open listings if a user turned it on, so the threshold has to be chosen per role bucket, not once.
- **The years asked climb with the bucket:** the median required is 3 years in IT support and marketing specialists, 4 in business and data analysis, 5 in most buckets and 6 in IT project management; 5+ required runs from 14% (IT support) to 49% (IT project management), and 2 years or fewer from 23% (IT support) down to 5.1% (IT project management).
- **EDA_SDA_1's degree rule fired on 3.7% of listings; the rebuilt rule finds a degree on a requirement line in 52%,** another 11% only on preferred lines, and 63% anywhere. 29% accept "or equivalent experience", and 29% require a degree with no equivalent (the "I have no degree" filter).
- **Most requirements are a bachelor's:** of the highest level named on a requirement line, 34% are bachelor, 9.5% master, 3.6% PhD and 0.90% associate; 52% name none.
- **The degree blocker is lowest in IT support (20% with no equivalent) and highest in quality assurance (37%),** with business and data analysis and data science at 36%.
- **These are unvalidated keyword rules:** the section 5 samples will not be marked (decided 2026-10-13), so these rates are firing rates, not precision. They label listings and drive opt-in filters; nothing hides by default.

## 4. Other text features

Each feature is a keyword rule on the cleaned text (`src/text_features.py`), reported with its firing rate and not hand-validated (decided 2026-10-13); it labels listings and can be an opt-in filter, and hides nothing by default. For each: how often it fires, and whether it relates to how fast listings close **beyond the confounders** (the 7-day close rate's gap with vs. without the feature, within company volume band × source type cells, on original listings with a full week of follow-up; EDA_SDA_4 tests the survivors properly). Listings where a feature is unknown (NA) are left out of that feature's comparison, not counted as "without"; the `known` column in 4b shows each feature's coverage (1 = every listing in the comparison is known).

| Group | Features |
|:--|:--|
| Labels and opt-in filters for some applicants | `clearance_required` (vs. `clearance_obtainable`), `no_sponsorship`, `us_citizen_required`, `cert_required` and named certifications, `shift_or_oncall`, `travel_pct`, `contract_to_hire` |
| What a newcomer is offered | `entry_wording`, `changer_wording` (now also bootcamp graduates, self-taught) |
| Hiring difficulty and urgency | `sign_on_bonus`, `hiring_immediately`, `relocation_offered`, `urgency_context` ("urgent care" no longer counts) |
| Evergreen wording (ghost signal) | six phrase groups and `evergreen_any`, compared with EDA_SDA_2's likely-ghost flag |
| Remote and length | `remote_mention` (minus "remote patient monitoring" and the like), `words_clean` |

In [ ]:
# 4a How often each text feature fires, overall and by role bucket (listings with a description)
FLAGS = [c for c in tf.FLAG_COLS if c not in ("degree_mentioned", "degree_required", "degree_preferred", "degree_equivalent")]
d = jobs[has_text]
rates = d[FLAGS].astype(float).mean().sort_values(ascending=False)
show("4a Share of listings with a description where each feature fires", rates.rename("share").round(4).to_frame())
show_shaded("4a Text features by role bucket (share of listings; features firing on 0.5%+)",
            d.groupby("role_bucket")[[c for c in FLAGS if rates[c] >= 0.005]].mean().T.round(3))

In [ ]:
# 4b Does each feature go with faster or slower closing, beyond the confounders? Original listings with a full week of
# follow-up; the 7-day close rate with vs. without the feature, raw and within volume band × source type cells
# (weighted by the cell's listings). A first look: EDA_SDA_4's adjusted model decides.
cut = pd.Timestamp("2026-09-23", tz=jobs.first_seen.dt.tz)
o = jobs[has_text & ~jobs.is_repost & (jobs.first_seen <= cut)].copy()
o["closed_7d"] = (~o.live & (o.days_open <= 7)).astype(float)
o["cell"] = o.volume_band.astype(str) + " | " + o.source_type.astype(str)

def within_gap(flag):
    k = o[o[flag].notna()]  # unknown (NA) listings sit out this feature's comparison
    g = k.groupby(["cell", k[flag].astype(bool)]).closed_7d.agg(["mean", "size"]).unstack()
    g = g.dropna()
    g = g[(g[("size", True)] >= 200) & (g[("size", False)] >= 200)]
    if g.empty:
        return np.nan
    w = g[("size", True)] + g[("size", False)]
    return float(((g[("mean", True)] - g[("mean", False)]) * w).sum() / w.sum())

rows = {}
for f in FLAGS + ["degree_required", "degree_equivalent"]:
    k = o[o[f].notna()]  # unknown (NA) listings sit out, rather than counting as "without"
    on = k[f].astype(bool)
    if on.sum() < 500:
        continue
    rows[f] = {"share": on.mean(), "known": len(k) / len(o),
               "closed in 7 days (with)": k.closed_7d[on].mean(), "closed in 7 days (without)": k.closed_7d[~on].mean(),
               "raw gap (points)": 100 * (k.closed_7d[on].mean() - k.closed_7d[~on].mean()), "gap within confounders (points)": 100 * within_gap(f)}
show("4b Text features vs. closing within 7 days (original listings with a description and a full week of follow-up)",
     pd.DataFrame(rows).T.sort_values("gap within confounders (points)").round(3) if rows
     else pd.DataFrame({"note": ["no feature fires on 500+ of these listings"]}))

In [ ]:
# 4c Evergreen wording vs. EDA_SDA_2's ghost signals: does it add to "old and from a quiet company"?
d = jobs[has_text]
grp = pd.Series(np.where(d.likely_ghost, "likely ghost", np.where(d.stale, "open 90+ days, company still posting",
                np.where(d.live, "open, 90 days or younger", "closed"))), index=d.index)
show("4c Evergreen wording by ghost status (share of listings with a description)",
     d.groupby(grp)[tf.EVERGREEN + ["evergreen_any"]].mean().round(4))

**What the data shows**
- **Most features are rare.** Remote is mentioned in 29%, a certification in 11%, a required clearance in 9.8% and no sponsorship in 6.9%; shift or on-call work 5.2%, US citizenship required 4.8%, entry-level wording 3.8%. Career-changer wording is on only 0.45% of listings, and a required certification on 0.91%.
- **Role buckets differ where you would expect:** certifications are mentioned in 40% of security listings (CompTIA 17%, security certifications 33%), 22% of IT support (CompTIA 17%) and 20% of IT project management (project certifications 15%); clearance is required in 26% of security, 15% of quality assurance and 14% of networks and systems listings, but 1.3% of technical sales. Shift or on-call work is 13% of IT support.
- **The 7-day close-rate gaps are modest.** About 21% of the original listings close within a week. Within company volume band × source type cells, the gaps run from -5.6 points (always-hiring wording, 2.1% of listings) to +5.4 points (urgency in context, 0.20%). Next are sign-on bonus (-3.9), any evergreen wording (-3.0), a required degree (-2.8) and Cisco certifications (-2.8). Features on 0.5% of listings or fewer are noisy.
- **Some raw gaps change sign once the confounders are held fixed:** contract-to-hire goes from +4.3 to -1.0 points and no-opening wording from -4.6 to +2.9. This is a first look, not a causal estimate; EDA_SDA_4's adjusted model decides which survive. In this run no listing with a description has an unknown feature, so the `known` share is 1 for every row.
- **Evergreen wording does not line up with the ghost flag:** any evergreen phrase is on 4.0% of closed listings, 3.3% of likely ghosts, 4.5% of listings open 90+ days from companies still posting and 4.2% of open listings 90 days or younger. As it stands it does not add to "old and from a quiet company"; always-hiring wording (2.1% of listings) is the largest of the six groups.

## 5. Validation samples (not being marked)

**Decided 2026-10-13: these samples will not be hand-checked.** 5a still writes a **sample** for each rule: 60 listings where it fires and 60 where it doesn't, stratified by role bucket, to `data/clean/eda3_validation/<rule>.csv` (local only), and 5b would read marks back and report precision and recall if anyone ever fills them in. Nobody is marking them, so **the rules are reported as keyword rules with their firing rates (sections 3, 4 and 7), not as validated blockers.** That sets what each rule may do: the text blockers (years asked, degree required with no equivalent, clearance, no sponsorship, required certification and the other career-change flags) are **labels on each listing and opt-in filters a career changer turns on, never hidden by default**. Only reliable rules hide by default (seniority, likely ghosts, occupations that aren't a tech path, all decided in EDA_SDA_2). Every rule reports how many listings it would hide (7a). Listing text never leaves the laptop.

In [ ]:
# 5a Write the hand-check samples (only if they don't exist yet, so marks are never overwritten). Shows counts only.
# Not being marked (decided 2026-10-13): kept so the rules could still be checked later.
RULES = ["years_required_5plus", "degree_required", "degree_equivalent", "clearance_required", "no_sponsorship", "cert_required",
         "entry_wording", "changer_wording", "evergreen_any", "urgency_context"]
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
d = jobs[has_text].assign(years_required_5plus=jobs.years_required.astype(float) >= 5)
written = {}
for rule in RULES:
    path = VALIDATION_DIR / f"{rule}.csv"
    if path.exists():
        continue
    picks = []
    for value in (True, False):  # about 60 matches and 60 non-matches, spread across role buckets by their size
        pool = d[d[rule].fillna(False).astype(bool) == value]
        if len(pool):
            picks.append(pool.groupby("role_bucket").sample(frac=min(60 / len(pool), 1), random_state=42).assign(fires=value))
    sample = pd.concat(picks)
    sample.assign(rule=rule, correct="")[["id", "rule", "fires", "role_bucket", "correct"]].to_csv(path, index=False)
    written[rule] = len(sample)
show("5a Hand-check samples written (open each CSV locally with the listing text from eda3_desc_clean.parquet)",
     pd.Series(written, name="rows").to_frame() if written else pd.DataFrame({"note": ["all samples already exist"]}))

In [ ]:
# 5b Read the marks back: precision of each rule (and recall from the non-matches), if anyone ever fills them in
# (not planned: decided 2026-10-13, the rules stay unvalidated labels and opt-in filters)
res = {}
for path in sorted(VALIDATION_DIR.glob("*.csv")):
    v = pd.read_csv(path)
    v = v[v.correct.isin([0, 1, "0", "1"])].assign(correct=lambda x: x.correct.astype(int))
    if v.empty:
        continue
    hit, miss = v[v.fires], v[~v.fires]
    res[path.stem] = {"marked": len(v), "precision": hit.correct.mean() if len(hit) else np.nan,
                      "share of non-matches that are right": miss.correct.mean() if len(miss) else np.nan,
                      "hides listings (precision 90%+)": bool(len(hit) and hit.correct.mean() >= 0.9)}
show("5b Validation results", pd.DataFrame(res).T if res else pd.DataFrame({"note": ["no marks: the samples are not being marked (decided 2026-10-13), so the rules stay unvalidated"]}))

**What the data shows**
- **No rule is validated, and none will be:** the samples are written (about 120 rows per rule, for ten rules: 5+ years required, degree required, degree equivalent, clearance, no sponsorship, required certification, entry wording, career-changer wording, evergreen wording, urgency) but are not being marked (decided 2026-10-13), so 5b reports no precision or recall.
- **So the rules are keyword rules reported with their firing rates:** they are labels and opt-in filters, never hidden by default; the 90% precision bar that would have let a rule hide listings is not applied to any of them.

## 6. Skills mapped to O*NET

**Question:** which tools and skills does each listing ask for, and how do they line up with what O*NET says the occupation needs? The answer feeds `ML_Fit`'s skills-based matching and the retraining compass's "skills that carry over".

Data: **O*NET 31.0 Database, USDOL/ETA (CC BY 4.0)**, downloaded with `python -m fetch.get_onet` into `data/external/onet/`. Release 31.0 splits skills into Essential Skills, Transferable Skills and Software Skills; `src/onet.py` reads them (importance `IM` and level `LV` ratings only; suppressed and not-relevant ratings dropped).

| Step | What | Output |
|:--|:--|:--|
| 6a | **Skill vocabulary:** every software tool O*NET lists (the Workplace Example, with its hot-technology and in-demand flags) plus the skill element names, lowercased with punctuation removed and common aliases folded together ("js" to "javascript", "MS Excel" to "Microsoft Excel"; C++, C# and .NET keep their meaning). Matched as **whole phrases**, so "java" is not found inside "javascript". Everyday words that are also tool names ("Go", "R", "Word", "Access", ...) are left out, because matching them as phrases fires on ordinary sentences | The vocabulary and a compiled matcher |
| 6b | **Extraction from requirement lines only** (a line under a requirements or preferred heading, or with a requirement cue such as "years", "experience", "proficiency"; responsibilities are not read), batch by batch from `eda3_desc_clean.parquet` with the same crash-safe `.part` writes as 2a-2b. Each match is marked **required** (on any required line) or **preferred** | `eda3_skills.parquet`: `id`, `name`, `kind` (tool or skill), `section`, `element_id` (cached; delete to rebuild) |
| 6c | **Coverage** and the most-asked tools and skills per role bucket | Aggregates |
| 6d | **Agreement with the listing's own occupation:** the share of a listing's matched skills that O*NET lists for its O*NET-SOC code | Aggregates |
| 6e | **Job Zone per bucket** (O*NET's 1-5 scale of preparation needed) as an accessibility prior | Aggregates |
| 6f | **Transferable skills** shared between the team's background occupations and each target bucket: the compass's "skills that carry over" | Aggregates |

Limits: a skill element name such as "Programming" or "Writing" is also an ordinary word, so skill matches are looser than tool matches; the keyword rules are not hand-checked (as in section 4), so they are labels and matching features, never filters that hide listings.

In [ ]:
# 6a Load the O*NET tables (data/external/onet/ from fetch.get_onet) and build the skill vocabulary and matcher
from src import onet
SKILLS_PATH = CLEAN_DIR / "eda3_skills.parquet"  # id, name, kind, section, element_id (cache; delete to rebuild)
essential, transferable = onet.ratings("Essential Skills"), onet.ratings("Transferable Skills")
software_tbl, zones = onet.software(), onet.job_zones()
vocab = onet.vocabulary(software_tbl, [essential, transferable])
matcher = onet.Matcher(vocab)
show("6a Skill vocabulary (O*NET 31.0 Database, USDOL/ETA, CC BY 4.0)", pd.DataFrame({"value": {
    "O*NET occupations with essential-skill ratings": f"{essential.code.nunique():,}",
    "software tools (Workplace Examples) kept": f"{(vocab.kind == 'tool').sum():,}",
    "  of them hot technologies": f"{vocab.hot[vocab.kind == 'tool'].sum():,}",
    "  of them in demand": f"{vocab.in_demand[vocab.kind == 'tool'].sum():,}",
    "skill element names kept": f"{(vocab.kind == 'skill').sum():,}",
    "names left out (very short or everyday words)": f"{vocab.attrs['skipped']:,}",
}}))

In [ ]:
# 6b Find each listing's tools and skills in its requirement lines, batch by batch on several processes (cached: delete
# eda3_skills.parquet to rebuild; the cache is rebuilt if the O*NET vocabulary or the cleaned text changed)
if onet.cache_valid(SKILLS_PATH, CLEAN_PATH, matcher):
    print(f"loaded the skills from {SKILLS_PATH.name} (delete it to rebuild 6b)")
else:
    start = time.time()
    n_rows = onet.extract(CLEAN_PATH, matcher, SKILLS_PATH,
                          progress=lambda n, k: print(f"  batch {n + 1}: {k:,} skill rows, {time.time() - start:.0f}s", end="\r"))
    print(f"\nfound {n_rows:,} skill rows in {(time.time() - start) / 60:.1f} minutes")
sk = pd.read_parquet(SKILLS_PATH, columns=["id", "name", "kind", "section"])
for c in ["name", "kind", "section"]:
    sk[c] = sk[c].astype("category")
sk = sk.merge(jobs[["id", "role_bucket"]], on="id", how="left")
print(f"{len(sk):,} skill rows over {sk.id.nunique():,} listings; {(sk.section == 'required').mean():.0%} of rows are on required lines")

In [ ]:
# 6c Coverage (share of listings with a description that have at least one matched skill) and the most-asked
# tools and skills per role bucket
d = jobs[has_text]
cov = d.assign(any_skill=d.id.isin(sk.id), required_skill=d.id.isin(sk.id[sk.section == "required"]),
               any_tool=d.id.isin(sk.id[sk.kind == "tool"])).groupby("role_bucket")[["any_skill", "required_skill", "any_tool"]].mean()
cov.loc["all"] = d.id.isin(sk.id).mean(), d.id.isin(sk.id[sk.section == "required"]).mean(), d.id.isin(sk.id[sk.kind == "tool"]).mean()
per_listing = sk.groupby("id", observed=True).size()
show("6c Coverage: share of listings with a description (by role bucket)", cov.rename(columns={
    "any_skill": "at least one skill", "required_skill": "one on a required line", "any_tool": "at least one tool"}).round(3))
print(f"matched skills per listing that has any: median {per_listing.median():.0f}, 90th percentile {per_listing.quantile(0.9):.0f}")
n_bucket = d.groupby("role_bucket").size()
for kind, k in [("tool", 20), ("skill", 15)]:
    counts = sk[sk.kind == kind].groupby(["name", "role_bucket"], observed=True).id.nunique().unstack(fill_value=0)
    counts = counts.reindex(columns=n_bucket.index, fill_value=0)
    top = counts.sum(axis=1).nlargest(k).index
    show_shaded(f"6c Most-asked {kind}s: share of each bucket's listings that ask for it (top {k} overall)", (counts.loc[top] / n_bucket).round(3))
hot = vocab[vocab.kind == "tool"].set_index("name")[["hot", "in_demand"]]
t = sk[sk.kind == "tool"]
show("6c Share of tool mentions that are O*NET hot technologies or in demand", pd.Series({
    "hot technology": t.name.map(hot.hot).astype(float).mean(), "in demand": t.name.map(hot.in_demand).astype(float).mean()}, name="share").round(3))

In [ ]:
# 6c Chart: coverage by role bucket
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(cov))
ax.barh(y - 0.2, 100 * cov.any_skill, height=0.4, color=MUTED, label="at least one skill or tool")
ax.barh(y + 0.2, 100 * cov.required_skill, height=0.4, color=BLUE, label="one on a required line")
ax.set(yticks=y, yticklabels=cov.index, xlabel="% of listings with a description", title="Listings with a matched O*NET skill or tool, by role bucket")
ax.legend(fontsize=8, frameon=False)
finish()

In [ ]:
# 6d Agreement with the listing's own occupation: of the skills matched in a listing, the share O*NET lists for its
# O*NET-SOC code (tools from Software Skills, skills from the Essential Skills importance ratings)
HAVE_CODE = "onet_code" in pq.ParquetFile(PART2_TABLE).schema.names
if not HAVE_CODE:
    print("onet_code is not in the scope table: save it from EDA_SDA_2 to run 6d-6f")
else:
    codes = pd.read_parquet(PART2_TABLE, columns=["id", "onet_code"]).set_index("id").onet_code
    listed = onet.listed_by_code(codes.dropna().unique(), software_tbl, essential)
    rows = {}
    for label, part in [("tools and skills", sk), ("tools only", sk[sk.kind == "tool"]), ("skills only", sk[sk.kind == "skill"])]:
        g = onet.agreement(part, codes, listed).join(jobs.set_index("id").role_bucket)
        rows[label] = g.groupby("role_bucket").share.median()
        rows[label].loc["all"] = g.share.median()
    show("6d Median share of a listing's matched skills that O*NET lists for its own code (by role bucket)", pd.DataFrame(rows).round(3))
    g_all = onet.agreement(sk, codes, listed)
    print(f"{len(g_all):,} listings have a code and a match; {(g_all.n_listed == 0).mean():.1%} have none of their matches on O*NET's list for their code")

In [ ]:
# 6e Job Zone per role bucket (1 little preparation needed ... 5 extensive) as an accessibility prior
if HAVE_CODE:
    jz = onet.job_zone_for(codes.reindex(jobs.id).reset_index(drop=True), zones)
    jz.index = jobs.index
    by_zone = pd.crosstab(jobs.role_bucket, jz.round().astype("Int64"), normalize="index")
    by_zone.columns = [f"zone {c}" for c in by_zone.columns]
    by_zone["mean zone"] = jz.groupby(jobs.role_bucket).mean()
    show_shaded("6e Job Zone per role bucket (share of listings; last column the mean)", by_zone.round(3))
    print(f"{jz.isna().mean():.1%} of listings have no Job Zone (no O*NET code, or a code O*NET does not list)")

In [ ]:
# 6f Skills that carry over: transferable skills (importance 3+ of 5) that the team's background occupations and each
# target bucket both need. BACKGROUND_OCCUPATIONS is the team's list of backgrounds, as O*NET-SOC codes (7 or 10
# characters; a name can list several). Left empty here: Jacob fills it in with the team's own backgrounds.
BACKGROUND_OCCUPATIONS = {}  # for example {"Accountant": ["13-2011"]}
if HAVE_CODE:
    matrix = onet.skill_matrix(transferable)
    code10 = jobs.assign(code=codes.reindex(jobs.id).to_numpy()).dropna(subset=["code"])
    weights = code10.groupby(["role_bucket", "code"]).size()
    targets = {b: onet.profile(matrix, list(w.droplevel(0).index), dict(w.droplevel(0))) for b, w in weights.groupby(level=0)}
    show("6f Each bucket's most important transferable skills (mean importance 1-5, weighted by its listings)",
         pd.DataFrame({b: pd.Series([f"{k} ({v:.2f})" for k, v in p.dropna().nlargest(6).items()]) for b, p in targets.items()}).fillna(""))
    if BACKGROUND_OCCUPATIONS:
        shared = {name: {b: onet.carry_over(onet.profile(matrix, codes_), p) for b, p in targets.items()} for name, codes_ in BACKGROUND_OCCUPATIONS.items()}
        show_shaded("6f Number of transferable skills both rate 3+: background (rows) by target bucket (columns)",
                    pd.DataFrame({b: {n: len(s[b]) for n, s in shared.items()} for b in targets}))
        show("6f The three strongest shared skills per background and bucket", pd.DataFrame(
            {b: {n: ", ".join(s[b].index[:3]) for n, s in shared.items()} for b in targets}))
    else:
        print("BACKGROUND_OCCUPATIONS is empty: fill it in to see which skills carry over from each background")

**What the data shows**
- (fill in after the run)

## 7. Final table

One row per scope listing (`id`), the text features from sections 3–4, and **text-rule flags** for the app (the `blocker_*` columns). They are unvalidated keyword rules (decided 2026-10-13): the app shows them as labels and opt-in filters (for example "I have no degree"), never hides by default, and 7a reports how many listings each would hide. Blank means unknown: the listing has no description. Saved as `text_table.parquet` for EDA_SDA_4 (joined by `id`) and `ML_Fit`.

In [ ]:
# 7a Text blockers: labels and opt-in filters in the app, never hidden by default (unvalidated keyword rules,
# decided 2026-10-13), and what each would hide among open listings if a user turned it on
out = jobs[["id"] + tf.FEATURE_COLS].copy()
known = jobs.words_raw.fillna(0) > 0
out["blocker_years_5plus"] = (jobs.years_required.astype(float) >= 5).where(known)  # a filter anyone can turn on (no years stated = False)
out["blocker_degree_no_equivalent"] = (jobs.degree_required & ~jobs.degree_equivalent).where(known)  # per applicant
out["blocker_clearance"] = jobs.clearance_required.where(known)                 # per applicant
out["blocker_no_sponsorship"] = jobs.no_sponsorship.where(known)                # per applicant (needs sponsorship)
out["blocker_cert_required"] = jobs.cert_required.where(known)                  # per applicant
BLOCKERS = [c for c in out if c.startswith("blocker_")]
show("7a Candidate blockers: share of listings with a description, and of open ones", pd.DataFrame({
    "all listings": out.loc[known, BLOCKERS].astype(float).mean(),
    "open on September 30": out.loc[known & jobs.live, BLOCKERS].astype(float).mean(),
}).round(3))
out.to_parquet(PART3_TABLE, index=False)
show("7a The final text table", pd.DataFrame({"value": {
    "listings": f"{len(out):,}", "columns": out.shape[1], "with text features": f"{known.mean():.1%}",
    "saved": f"{PART3_TABLE.name} ({PART3_TABLE.stat().st_size / 1e9:.2f} GB)",
}}))
print(f"[EDA_SDA_3 done in {(time.time() - _t0) / 60:.1f} min]")

## 8. Conclusion

**Summary**
- **Cleanup:** 908,291 scope listings, 82.4% with a description; the median description falls from 735 words to 573 after boilerplate removal (85–86% of text kept per source type). Equal opportunity and legal text is the largest share removed (6.6% of all words).
- **Rebuilt rules (unvalidated keyword rules):** 5+ years on a requirement line is 35.6% of listings (36% of open ones), against 33.7% for EDA_SDA_1's rule; a degree is required on 52% (EDA_SDA_1: 3.7%), 29% with no equivalent accepted.
- **Other features:** most are rare (remote 29%, certification 11%, clearance 9.8%, no sponsorship 6.9%, career-changer wording 0.45%). Their 7-day close gaps within confounders stay within about 5.6 points either way.
- **Evergreen wording** does not concentrate in likely-ghost listings (3.3%, against 4.0–4.5% elsewhere).

**Answers to the clarifying questions**
- *Length: intent or boilerplate?* Not answered here; the cleanup shows boilerplate is a large part of the text, and length (`words_clean`) is saved for EDA_SDA_4.
- *Urgency in context vs. close time?* It fires on 0.20% of listings, with a +5.4-point gap within confounders: too few listings to rely on.
- *Evergreen wording as a ghost signal?* Not as it stands (see above).
- *Are EDA_SDA_1's flags accurate?* The years rule misses some (the new rule finds more in 12% of listings) and the degree rule missed most requirements (3.7% against 52%). Precision is untested: the hand-check samples will not be marked (decided 2026-10-13).
- *Which requirements hide a listing?* None by default. Years asked, clearance, no sponsorship, required certification and a degree with no equivalent are **labels and opt-in filters**, and would hide this much if a user turned each on: 5+ years required (35.6% of listings), clearance (9.8%), no sponsorship (6.9%), a required certification (0.91%) and a degree with no equivalent (29%). **None is validated (the section 5 samples will not be marked, decided 2026-10-13), so none hides by default.**
- *Skills mapped to O\*NET?* Pending the O\*NET download (section 6).

**Limits:** the gaps in 4b are a first look and associational, not causal; the rules are unvalidated keyword rules (firing rates, no precision); the O\*NET skills and the length question are open.

**Handoff:** `text_table.parquet` (908,291 listings, 45 columns, text features and text-rule flags (labels, unvalidated), joined by `id`; blank where there is no description) for EDA_SDA_4 and `ML_Fit`; `eda3_desc_clean.parquet` (cleaned text) and `eda3_lemmas_sample.parquet` (150,001 listings) for `ML_Fit`.